# 📘 Groww API — Learn & Explore (No Trading Yet)

This notebook is your **learning ground**. It covers:
1. How authentication works step by step
2. How to verify your connection is live
3. How to read your account details (funds, margins, holdings)
4. How to explore market data (live price, historical candles)
5. How to inspect your orders & positions (read-only)

> ✅ **Nothing in this notebook places a real order.** It is 100% read-only and safe to run.

---

## 🔧 STEP 0 — Setup: Imports & Path

Always run this cell first. It tells Python where to find your `src/` folder and loads your `.env` credentials.

In [ ]:
import sys
import os

# So Python can find src/groww_helper.py from the notebooks/ folder
sys.path.append(os.path.abspath('../'))

from dotenv import load_dotenv
load_dotenv('../.env')  # Loads GROWW_API_KEY, GROWW_API_SECRET, GROWW_TOTP_SECRET

print("✅ Setup complete.")
print("API Key loaded:", "YES" if os.getenv('GROWW_API_KEY') else "❌ NOT FOUND — check your .env file")
print("TOTP Secret loaded:", "YES" if os.getenv('GROWW_TOTP_SECRET') else "Not set (will try Key+Secret flow)")

---
## 🔐 STEP 1 — Authentication: Get a Live Client

The `GrowwHelper.get_api_client()` function from `src/groww_helper.py` handles everything:
- It reads your `.env` file
- If you have `GROWW_TOTP_SECRET`, it auto-generates the OTP and logs in (recommended)
- Otherwise it uses `GROWW_API_KEY` + `GROWW_API_SECRET`
- Returns a `groww` client object you use for all API calls

Think of `groww` like a remote control for your Groww account.

In [ ]:
from src.groww_helper import GrowwHelper

try:
    groww = GrowwHelper.get_api_client()
    print("\n🎉 Client created successfully! You are connected to Groww.")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("👉 Check that your .env file has the correct credentials.")

---
## ✅ STEP 2 — Connection Check: Is It Actually Live?

Just creating the client doesn't guarantee you're connected. We verify by making a real API call.
Fetching the order list is a lightweight, read-only call — perfect for a connection health check.

In [ ]:
# --- Connection Health Check ---
# get_order_list() fetches your existing orders from Groww's servers.
# If this works, your API key is valid and the session is live.
# timeout=5 means: wait up to 5 seconds for a response.

try:
    orders = groww.get_order_list(timeout=5)
    print("✅ CONNECTION IS LIVE!")
    print(f"   Orders found: {len(orders) if orders else 0}")
    print("   Raw response:", orders)
except Exception as e:
    print(f"❌ Connection check failed: {e}")
    print("   Possible reasons: expired token, wrong API key, or network issue.")

---
## 💰 STEP 3 — Account: Check Your Funds & Margins

**Margin** = How much money you have available to trade.
- `available_margin` = Cash you can use RIGHT NOW to buy stocks
- `used_margin` = Cash already locked in open positions
- `total_margin` = available + used

This is the most important thing to check before doing anything.

In [ ]:
# --- Check available funds/margin ---
try:
    margin = groww.get_margin(timeout=5)
    print("💰 MARGIN / FUNDS SUMMARY")
    print("-" * 35)
    print(margin)  # Print raw first so you can see what keys are returned
except Exception as e:
    print(f"❌ Could not fetch margin: {e}")

In [ ]:
# --- Pretty print margin if it's a dict ---
# Once you know the keys from above, use them here:
import json

try:
    margin = groww.get_margin(timeout=5)
    if isinstance(margin, dict):
        print("💰 MARGIN BREAKDOWN:")
        for key, val in margin.items():
            print(f"   {key}: {val}")
    else:
        print(margin)
except Exception as e:
    print(f"❌ Error: {e}")

---
## 📦 STEP 4 — Holdings: What Stocks Do You Already Own?

**Holdings** = Stocks you have bought and are sitting in your Demat account (long-term, not just today's trades).

Example: If you bought 5 shares of INFY last month, they show up in holdings.

In [ ]:
# --- Fetch your current Demat holdings ---
try:
    holdings = groww.get_holdings(timeout=5)
    print("📦 YOUR HOLDINGS")
    print("-" * 40)
    print(holdings)
except Exception as e:
    print(f"❌ Could not fetch holdings: {e}")

In [ ]:
# --- Display holdings as a clean table using pandas ---
import pandas as pd

try:
    holdings = groww.get_holdings(timeout=5)
    if holdings:
        df = pd.DataFrame(holdings)
        print(f"Total stocks held: {len(df)}")
        display(df)  # 'display' works in Jupyter and renders a nice table
    else:
        print("No holdings found. Your Demat account may be empty.")
except Exception as e:
    print(f"❌ Error: {e}")

---
## 📋 STEP 5 — Orders: View All Past & Open Orders

**Orders** = Any buy/sell instructions you've sent, whether pending, executed, or cancelled.

Useful for understanding what's happened in your account.

In [ ]:
# --- View your order history ---
try:
    orders = groww.get_order_list(timeout=5)
    print(f"📋 ORDERS FOUND: {len(orders) if orders else 0}")
    print("-" * 40)
    if orders:
        df_orders = pd.DataFrame(orders)
        display(df_orders)
    else:
        print("No orders yet.")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# --- Filter only OPEN / PENDING orders ---
try:
    orders = groww.get_order_list(timeout=5)
    if orders:
        df_orders = pd.DataFrame(orders)
        # Look for a 'status' column — adjust column name based on what you saw above
        if 'status' in df_orders.columns:
            open_orders = df_orders[df_orders['status'].str.upper() == 'OPEN']
            print(f"Open/pending orders: {len(open_orders)}")
            display(open_orders)
        else:
            print("Column names available:", df_orders.columns.tolist())
    else:
        print("No orders found.")
except Exception as e:
    print(f"❌ Error: {e}")

---
## 📊 STEP 6 — Positions: Today's Open Trades

**Positions** = Stocks you bought and sold **today** (intraday). Different from Holdings.
- Holdings = long-term stocks in Demat
- Positions = today's open trades (not yet settled)

In [ ]:
# --- Check today's open positions ---
try:
    positions = groww.get_positions(timeout=5)
    print(f"📊 OPEN POSITIONS TODAY: {len(positions) if positions else 0}")
    if positions:
        df_pos = pd.DataFrame(positions)
        display(df_pos)
    else:
        print("No open positions today.")
except Exception as e:
    print(f"❌ Error: {e}")

---
## 📡 STEP 7 — Live Market Data: Get the LTP of a Stock

**LTP** = Last Traded Price. The most recent price at which a stock was bought/sold on the exchange.

This uses `GrowwFeed` — a separate live data stream client (different from `GrowwAPI`).

Think of `GrowwAPI` as your account manager and `GrowwFeed` as your live market data TV channel.

In [ ]:
from growwapi import GrowwFeed, GrowwAPI
import os

api_key = os.getenv('GROWW_API_KEY')

# --- Create a feed client ---
# GrowwFeed connects to a live data stream (like a WebSocket)
feed = GrowwFeed(api_key)
print("✅ Feed client created.")

In [ ]:
# --- Subscribe to live data for a stock ---
# SEGMENT_CASH = NSE/BSE cash equity segment (regular stocks like SBIN, INFY, RELIANCE)
# Subscribe first, then get LTP. Subscribe = "tune in to this channel."

STOCK = "SBIN"  # Change to any NSE stock symbol you want to check

feed.subscribe_live_data(GrowwAPI.SEGMENT_CASH, STOCK)
print(f"📡 Subscribed to live feed for {STOCK}...")

# Now get LTP — waits up to 5 seconds for the data to arrive
ltp = feed.get_stocks_ltp(STOCK, timeout=5)
print(f"💹 LTP of {STOCK}: ₹{ltp}")

In [ ]:
# --- Check LTP for multiple stocks at once ---
WATCHLIST = ["RELIANCE", "TCS", "INFY", "SBIN", "HDFCBANK"]

print("📡 Subscribing to multiple stocks...")
for symbol in WATCHLIST:
    feed.subscribe_live_data(GrowwAPI.SEGMENT_CASH, symbol)

import time
time.sleep(3)  # Give 3 seconds for data to arrive

print("\n💹 LIVE PRICES:")
print("-" * 30)
for symbol in WATCHLIST:
    ltp = feed.get_stocks_ltp(symbol)  # No timeout — data already arrived
    print(f"   {symbol:15s}: ₹{ltp}")

---
## 📈 STEP 8 — Historical Data: Past Candles (OHLCV)

**Candle** = A summary of a stock's price movement in a time period.
Each candle has:
- **O** = Open price (price at start of period)
- **H** = High price (highest point)
- **L** = Low price (lowest point)
- **C** = Close price (price at end of period)
- **V** = Volume (how many shares were traded)

This is the raw material for building any trading strategy later.

In [ ]:
# --- Fetch historical daily candles ---
# interval options (check SDK docs for exact strings): '1m', '5m', '15m', '1h', '1d'
# NOTE: Exact method name may vary — check what your SDK version exposes

try:
    candles = groww.get_candle_data(
        symbol="SBIN",
        exchange="NSE",           # or 'BSE'
        interval="1d",            # daily candles
        from_date="2026-01-01",
        to_date="2026-06-01"
    )
    df_candles = pd.DataFrame(candles)
    print(f"✅ Got {len(df_candles)} candles for SBIN")
    display(df_candles.head(10))
except AttributeError:
    print("⚠️  Method name may be different. Let's check what's available...")
    print("Available methods on groww client:")
    methods = [m for m in dir(groww) if not m.startswith('_')]
    for m in methods:
        print(f"   groww.{m}()")

In [ ]:
# --- Explore ALL available methods on the groww client ---
# Run this if you're unsure what functions exist in the SDK version you have

print("🔍 All callable methods on your groww client:")
print("-" * 50)
methods = [m for m in dir(groww) if not m.startswith('_') and callable(getattr(groww, m))]
for m in methods:
    print(f"  groww.{m}()")

In [ ]:
# --- Explore ALL available methods on the feed client ---
print("🔍 All callable methods on your feed client:")
print("-" * 50)
feed_methods = [m for m in dir(feed) if not m.startswith('_') and callable(getattr(feed, m))]
for m in feed_methods:
    print(f"  feed.{m}()")

---
## 📉 STEP 9 — Plot Historical Data (Visual Check)

Once you have candle data working in Step 8, use this to plot it visually. A good sanity check that the data looks real.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Assuming df_candles from Step 8 is available
# Adjust column names to match what your SDK returns
try:
    df_plot = df_candles.copy()
    
    # Common column names — adjust if yours differ
    # Try: 'timestamp', 'date', 'time' for the x-axis
    # Try: 'close', 'c', 'Close' for the y-axis
    print("Columns in candle data:", df_plot.columns.tolist())
    
    # --- Adjust these two lines based on your actual column names ---
    x_col = 'timestamp'  # or 'date'
    y_col = 'close'       # or 'c' or 'Close'
    
    plt.figure(figsize=(14, 5))
    plt.plot(pd.to_datetime(df_plot[x_col]), df_plot[y_col], color='steelblue', linewidth=1.5)
    plt.title('SBIN — Daily Close Price', fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Price (₹)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not plot: {e}")
    print("Run Step 8 first to fetch candle data into df_candles.")

---
## 🔍 STEP 10 — Inspect a Specific Order (by Order ID)

Once you have order IDs from Step 5, you can get full details on any single order.

In [ ]:
# --- Get details of a specific order ---
# Replace with an actual order_id from your get_order_list() output above

ORDER_ID = "your_order_id_here"  # ← replace this

try:
    order_detail = groww.get_order_detail(ORDER_ID, timeout=5)
    print("📋 ORDER DETAILS:")
    print("-" * 40)
    if isinstance(order_detail, dict):
        for k, v in order_detail.items():
            print(f"   {k}: {v}")
    else:
        print(order_detail)
except Exception as e:
    print(f"❌ Error: {e}")

---
## 🧠 STEP 11 — Understanding What You Saw (Quick Reference)

| Term | What it means |
|---|---|
| **GrowwAPI** | Your account manager — orders, margin, holdings, positions |
| **GrowwFeed** | Your live market data stream — LTP, depth, tick-by-tick |
| **LTP** | Last Traded Price — current market price of a stock |
| **Margin** | Cash available to trade |
| **Holdings** | Stocks in your Demat (bought days/months ago, long-term) |
| **Positions** | Today's open trades (intraday, not yet settled) |
| **Orders** | Instructions sent to exchange — can be OPEN, EXECUTED, CANCELLED |
| **Candle / OHLCV** | Historical price summary per time period |
| **SEGMENT_CASH** | Regular NSE/BSE equity stocks |
| **timeout** | Seconds to wait for API response before giving up |
| **subscribe_live_data()** | Tunes into live price updates for a stock |
| **get_stocks_ltp()** | Gets the latest price after subscribing |

---

## ✅ What to Do Next

1. Run each step top to bottom. Read what comes out.
2. In Step 8-9, use Step 8's method discovery cell to find the exact method names your SDK version uses.
3. Once you're comfortable reading data, the next phase is building a signal (e.g., moving average crossover) — but that comes later.

> 💡 You are not ready to place orders yet — and that's the right call. Understand the data first.